# Station Stacking v2 - KMIA

Wide HRRR/GFS same-day 11am notebook for `KMIA`.

This workflow tunes XGBoost, LightGBM, and CatBoost with Optuna/TPE against RMSE on two fixed validation folds: train 2021-2023 validate 2024, train 2022-2024 validate 2025. It then trains on 2021-2025, tests on 2026, adds a Ridge meta-model stack, and simulates deterministic 2-degree weather brackets without calling Polymarket.

Version 2 adds notebook-level feature engineering and writes experiment artifacts to `data/calibration/station_stacking_v2`.


In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

STATION_ID = "KMIA"
FAST_MODE = False
OPTUNA_TRIALS = 100
STACK_OPTUNA_TRIALS = 50
OPTUNA_VERBOSE = True
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
from src.calibration.station_stacking import (
    StationStackingConfig,
    missing_model_dependencies,
    run_station_year_split_experiment,
)


## V2 Feature Engineering

Adds extra pre-model features from existing same-day and lagged signals. These features use only 11 AM observations, provider forecasts, and prior actual-high history, then patch the station-stacking dataset builder before the experiment runs.


In [3]:
import numpy as np
import pandas as pd
import src.calibration.station_stacking as station_stacking_module

V2_FEATURE_COLUMNS = [
    "v2_recent_heat_anomaly_f",
    "v2_recent_heat_momentum_f",
    "v2_morning_warmup_to_consensus_f",
    "v2_consensus_minus_7d_actual_f",
    "v2_spread_per_warmup_f",
    "v2_humidity_warmup_interaction",
]


def add_v2_feature_engineering(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    observed_temp = pd.to_numeric(out.get("observed_temp_at_as_of_f"), errors="coerce")
    observed_humidity = pd.to_numeric(out.get("observed_humidity_at_as_of"), errors="coerce")
    provider_mean = pd.to_numeric(out.get("provider_mean_high_f"), errors="coerce")
    provider_spread = pd.to_numeric(out.get("provider_spread_high_f"), errors="coerce")
    lag_1d = pd.to_numeric(out.get("actual_high_lag_1d"), errors="coerce")
    roll_7d = pd.to_numeric(out.get("actual_high_roll_7d_mean"), errors="coerce")
    roll_30d = pd.to_numeric(out.get("actual_high_roll_30d_mean"), errors="coerce")

    warmup_to_consensus = provider_mean - observed_temp
    out["v2_recent_heat_anomaly_f"] = lag_1d - roll_30d
    out["v2_recent_heat_momentum_f"] = roll_7d - roll_30d
    out["v2_morning_warmup_to_consensus_f"] = warmup_to_consensus
    out["v2_consensus_minus_7d_actual_f"] = provider_mean - roll_7d
    out["v2_spread_per_warmup_f"] = provider_spread / warmup_to_consensus.abs().clip(lower=1.0)
    out["v2_humidity_warmup_interaction"] = (observed_humidity / 100.0) * warmup_to_consensus
    return out


if not hasattr(station_stacking_module, "_v2_original_build_station_wide_dataset"):
    station_stacking_module._v2_original_build_station_wide_dataset = station_stacking_module.build_station_wide_dataset


def build_station_wide_dataset_v2(*args, **kwargs):
    frame = station_stacking_module._v2_original_build_station_wide_dataset(*args, **kwargs)
    return add_v2_feature_engineering(frame)


station_stacking_module.build_station_wide_dataset = build_station_wide_dataset_v2
V2_FEATURE_COLUMNS


['v2_recent_heat_anomaly_f',
 'v2_recent_heat_momentum_f',
 'v2_morning_warmup_to_consensus_f',
 'v2_consensus_minus_7d_actual_f',
 'v2_spread_per_warmup_f',
 'v2_humidity_warmup_interaction']

## Model Scores


In [4]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_verbose=OPTUNA_VERBOSE,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2",
)
result = run_station_year_split_experiment(config)
result.scoreboard


[I 2026-06-06 13:40:58,206] A new study created in memory with name: no-name-1ae5505d-3ae0-4fc0-baa9-9f4fa8a89eff
[I 2026-06-06 13:41:13,829] Trial 0 finished with value: 2.3416777553551738 and parameters: {'n_estimators': 799, 'learning_rate': 0.12369619597856178, 'max_depth': 6, 'min_child_weight': 2.385234757844707, 'gamma': 0.7800932022121826, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.5290418060840998, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.6677615511747083}. Best is trial 0 with value: 2.3416777553551738.
[I 2026-06-06 13:45:48,797] Trial 1 finished with value: 2.218367820143932 and parameters: {'n_estimators': 1440, 'learning_rate': 0.0032515743808034223, 'max_depth': 8, 'min_child_weight': 8.23143373099555, 'gamma': 1.0616955533913808, 'subsample': 0.5909124836035503, 'colsample_bytree': 0.5917022549267169, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.2922905212920093}. Best is trial 1 with value: 2.218367820143932.
[I 2026-06-06 13:46:32,963] Trial

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,664,1.373476,2.166584
1,validation_2024_2025,lightgbm,664,1.439081,2.211563
2,validation_2024_2025,catboost,664,1.383517,2.132923
3,validation_2024_2025,hrrr_raw,664,2.179532,2.905718
4,validation_2024_2025,gfs_raw,664,3.540756,4.037614
5,test_2026,xgboost,133,1.976977,2.879518
6,test_2026,lightgbm,133,1.911371,2.766686
7,test_2026,catboost,133,1.916566,2.701904
8,test_2026,ridge_stack,133,1.842646,2.629061
9,test_2026,hrrr_raw,133,2.554333,3.395461


## 2026 Weather Brackets


In [5]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bracket_accuracy_pct
0,xgboost,133,1.976977,2.879518,37.593985
1,lightgbm,133,1.911371,2.766686,39.097744
2,catboost,133,1.916566,2.701904,36.090226
3,ridge_stack,133,1.842646,2.629061,35.338346
4,hrrr_raw,133,2.554333,3.395461,21.052632
5,gfs_raw,133,4.164163,4.755602,6.766917


In [7]:
import pandas as pd


def adjacent_brackets(bracket):
    if pd.isna(bracket):
        return []
    text = str(bracket).strip()
    if not text or "-" not in text:
        return []
    try:
        lower = int(text.split("-", 1)[0])
    except ValueError:
        return []
    return [
        f"{lower - 2}-{lower - 1}",
        f"{lower}-{lower + 1}",
        f"{lower + 2}-{lower + 3}",
    ]


bracket_3way = result.bracket_predictions.copy()

valid = bracket_3way["actual_bracket"].notna() & bracket_3way["predicted_bracket"].astype(str).str.strip().ne("")
bracket_3way = bracket_3way.loc[valid].copy()
bracket_3way["picked_brackets"] = bracket_3way["predicted_bracket"].map(adjacent_brackets)
bracket_3way["three_bracket_hit"] = bracket_3way.apply(
    lambda row: row["actual_bracket"] in row["picked_brackets"],
    axis=1,
)

three_bracket_accuracy = (
    bracket_3way
    .groupby("method", as_index=False)
    .agg(
        count=("three_bracket_hit", "size"),
        exact_bracket_accuracy_pct=("bracket_hit", lambda x: x.mean() * 100),
        three_bracket_accuracy_pct=("three_bracket_hit", lambda x: x.mean() * 100),
    )
    .sort_values("three_bracket_accuracy_pct", ascending=False)
)

three_bracket_accuracy


,method,count,exact_bracket_accuracy_pct,three_bracket_accuracy_pct
0,catboost,133,36.090226,79.699248
4,ridge_stack,133,35.338346,79.699248
3,lightgbm,133,39.097744,78.195489
5,xgboost,133,37.593985,76.691729
2,hrrr_raw,133,21.052632,69.924812
1,gfs_raw,133,6.766917,32.330827
